# Notebook 1: Configuration

**Purpose:** Define all file paths, Chroma collection names, OpenAI parameters, and prompt templates used across the system.

**Execution order:** This notebook must be run first. All subsequent notebooks depend on the variables defined here.

**Assumed directory structure (relative to this notebook):**
```
RAG/
├── 01_config.ipynb          <- current file
├── 02_rag_system.ipynb
├── 03_herwell_chatbot.ipynb
├── chroma_db_user/
└── chroma_db_medical/
    └── chroma_db/
```

## 1.1 Path Verification

In [1]:
import os

# Set project root to the directory containing this notebook.
# If this fails, set BASE_DIR manually as an absolute path.
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
# BASE_DIR = r"D:\任梓嘉\NUS\sem2\Innovation Challenge\RAG"

paths_to_check = {
    "chroma_db_user/": os.path.join(BASE_DIR, "chroma_db_user"),
    "chroma_db_medical/chroma_db/": os.path.join(BASE_DIR, "chroma_db_medical", "chroma_db"),
}

print("Path verification:")
all_ok = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}]  {name}")
    print(f"             -> {path}")
    if not exists:
        all_ok = False

print()
print("All paths verified." if all_ok else "ERROR: One or more paths are missing. Resolve before proceeding.")

Path verification:
  [OK]  chroma_db_user/
             -> d:\任梓嘉\NUS\sem2\Innovation Challenge\RAG\chroma_db_user
  [OK]  chroma_db_medical/chroma_db/
             -> d:\任梓嘉\NUS\sem2\Innovation Challenge\RAG\chroma_db_medical\chroma_db

All paths verified.


## 1.2 Query Collection Names

In [2]:
import chromadb

user_db_path    = os.path.join(BASE_DIR, "chroma_db_user")
medical_db_path = os.path.join(BASE_DIR, "chroma_db_medical", "chroma_db")

print("=== chroma_db_user collections ===")
c_user = chromadb.PersistentClient(path=user_db_path)
print([col.name for col in c_user.list_collections()])

print()
print("=== chroma_db_medical collections ===")
c_medical = chromadb.PersistentClient(path=medical_db_path)
print([col.name for col in c_medical.list_collections()])

=== chroma_db_user collections ===
['population_summaries']

=== chroma_db_medical collections ===
['medical_knowledge']


## 1.3 Configuration Definitions

Collection names confirmed from the query above:
- User DB: `population_summaries`
- Medical DB: `medical_knowledge`

Update these values if your query returns different names.

In [3]:
import os
from dotenv import load_dotenv

# ------------------------------------------------------------
# File paths
# ------------------------------------------------------------
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
# BASE_DIR = r"D:\任梓嘉\NUS\sem2\Innovation Challenge\RAG"

# Load API key from .env in project root
load_dotenv(os.path.join(BASE_DIR, ".env"))

CHROMA_USER_DB_PATH    = os.path.join(BASE_DIR, "chroma_db_user")
CHROMA_MEDICAL_DB_PATH = os.path.join(BASE_DIR, "chroma_db_medical", "chroma_db")

# ------------------------------------------------------------
# Chroma collection names  (must match names used at ingestion time)
# ------------------------------------------------------------
USER_COLLECTION_NAME    = "population_summaries"
MEDICAL_COLLECTION_NAME = "medical_knowledge"

# ------------------------------------------------------------
# OpenAI configuration
# ------------------------------------------------------------
OPENAI_CHAT_MODEL      = "gpt-4o"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"   # output dimension: 1536

# ------------------------------------------------------------
# Retrieval parameters
# ------------------------------------------------------------
TOP_K_USER    = 3    # number of results to retrieve from the user vector store
TOP_K_MEDICAL = 5    # number of results to retrieve from the medical vector store

# ------------------------------------------------------------
# Risk-level system prompts
# risk=0: routine / no concern
# risk=1: low concern, general guidance
# risk=2: moderate concern, recommend professional consultation
# risk=3: emergency, immediate action required
# ------------------------------------------------------------
SYSTEM_PROMPTS = {
    0: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates this is a routine inquiry with no immediate health concern (risk level 0).
Provide accurate, evidence-based information in a calm and reassuring tone.
Encourage healthy lifestyle habits where relevant.
Always remind users that a healthcare professional can offer personalised advice.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    1: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates a low-level health concern that warrants attention (risk level 1).
Provide clear, evidence-based information and practical self-care suggestions.
Advise the user to monitor their symptoms and consult a healthcare professional if symptoms persist or worsen.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    2: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates a moderate health concern that requires professional evaluation (risk level 2).
Provide helpful background information while clearly recommending that the user schedule an appointment with a qualified healthcare provider soon.
Do not attempt to diagnose; focus on empowering the user with information to facilitate that consultation.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    3: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment has flagged this as a potential EMERGENCY (risk level 3).
Your response must first and foremost direct the user to seek immediate emergency medical care.
Keep your message clear, calm, and brief — do not overwhelm the user with information.
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",
}

# Convenience: risk-level-specific recommendation footers appended to responses
RISK_RECOMMENDATIONS = {
    0: "If you have further questions or your situation changes, feel free to ask. Stay well!",
    1: ("Please monitor your symptoms over the next 24–48 hours. "
        "If they persist or worsen, contact your healthcare provider."),
    2: ("We recommend booking an appointment with your healthcare provider as soon as possible. "
        "If symptoms worsen suddenly, seek urgent care or go to the nearest clinic."),
    3: ("⚠️ EMERGENCY: Please call emergency services immediately "
        "(e.g., 120 in China, 999 in Singapore, 911 in the US). "
        "If possible, inform someone nearby of your condition and proceed to the nearest emergency department. "
        "Do not wait."),
}

# ------------------------------------------------------------
# RAG prompt template  (risk level injected at query time)
# ------------------------------------------------------------
RAG_PROMPT_TEMPLATE = """Based on the following context, answer the user's health question.

=== Triage Assessment ===
Risk Level: {risk_level}
(This risk level was determined by analysing the user's personal physiological data
from their wearable device, including cycle patterns, HRV, temperature, and symptoms.)
{triage_result}

=== User's Description of Their Issue ===
The following is what the user has described about their current symptoms or concerns.
Take this seriously as it reflects the user's lived experience.
Ignore the language of this section.
{user_context}

=== Relevant Medical Knowledge ===
The following is extracted from authoritative medical guidelines
(ACOG, ESHRE, MedlinePlus, and peer-reviewed literature on Asian populations).
Use this to ground your recommendations in evidence-based information.
Ignore the language of this section.
{medical_context}

=== Response Instructions ===
1. Acknowledge the user's triage risk level as determined by their physiological data.
2. Address the user's described symptoms and concerns directly.
3. Ground your recommendations in the medical knowledge provided.
4. Synthesise all three sources: risk level + user description + medical guidelines.
5. Never diagnose. Never prescribe specific medications or dosages.
6. Adjust your tone and urgency based on the risk level:
   - Level 0: Reassuring, wellness-focused
   - Level 1: Supportive, practical self-care advice
   - Level 2: Concerned, recommend professional consultation
   - Level 3: Urgent, direct to emergency care immediately

=== User Question ===
{question}

Please provide a helpful, personalised response that integrates the user's risk profile,
their described concerns, and evidence-based medical guidance.
Respond ONLY in the same language as the user's question above,
regardless of the language used in the context sections.
{risk_recommendation}"""

# ------------------------------------------------------------
# Standalone emergency response (used when risk == 3 and no RAG is needed)
# ------------------------------------------------------------
EMERGENCY_RESPONSE = """⚠️ WARNING: The symptoms you have described may require IMMEDIATE medical attention.

Recommended actions:
1. Call emergency services immediately (e.g., 120 in China, 999 in Singapore, 911 in the US).
2. Proceed to the nearest emergency department if you are able to do so safely.
3. Inform someone nearby of your condition right now.

This AI assistant cannot substitute for emergency medical care. Please seek professional help immediately."""

# ------------------------------------------------------------
# Helper: return the correct system prompt for a given risk level
# ------------------------------------------------------------
def get_system_prompt(risk_level: int) -> str:
    """Return the system prompt corresponding to the triage risk level (0–3)."""
    return SYSTEM_PROMPTS.get(risk_level, SYSTEM_PROMPTS[0])

print("Configuration loaded successfully.")
print(f"  User DB path:        {CHROMA_USER_DB_PATH}")
print(f"  Medical DB path:     {CHROMA_MEDICAL_DB_PATH}")
print(f"  User collection:     {USER_COLLECTION_NAME}")
print(f"  Medical collection:  {MEDICAL_COLLECTION_NAME}")
print(f"  Chat model:          {OPENAI_CHAT_MODEL}")
print(f"  Embedding model:     {OPENAI_EMBEDDING_MODEL}")
print(f"  Risk-level prompts:  {list(SYSTEM_PROMPTS.keys())}")
print(f"  API key loaded:      {'YES' if os.environ.get('OPENAI_API_KEY') else 'NO — check .env file'}")


Configuration loaded successfully.
  User DB path:        d:\任梓嘉\NUS\sem2\Innovation Challenge\RAG\chroma_db_user
  Medical DB path:     d:\任梓嘉\NUS\sem2\Innovation Challenge\RAG\chroma_db_medical\chroma_db
  User collection:     population_summaries
  Medical collection:  medical_knowledge
  Chat model:          gpt-4o
  Embedding model:     text-embedding-3-small
  Risk-level prompts:  [0, 1, 2, 3]
  API key loaded:      YES


## 1.4 Configuration Verification

In [4]:
# Verify OpenAI API key
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print(f"OPENAI_API_KEY is set (prefix: {api_key[:8]}...)")
else:
    print("ERROR: OPENAI_API_KEY is not set.")
    print("  Set it with: import os; os.environ['OPENAI_API_KEY'] = 'sk-...'")

# Verify Chroma connectivity and collection existence
import chromadb

try:
    c = chromadb.PersistentClient(path=CHROMA_USER_DB_PATH)
    col = c.get_collection(USER_COLLECTION_NAME)
    print(f"User DB connection OK. Collection '{USER_COLLECTION_NAME}': {col.count()} records.")
except Exception as e:
    print(f"ERROR: User DB failed to load: {e}")

try:
    c2 = chromadb.PersistentClient(path=CHROMA_MEDICAL_DB_PATH)
    col2 = c2.get_collection(MEDICAL_COLLECTION_NAME)
    print(f"Medical DB connection OK. Collection '{MEDICAL_COLLECTION_NAME}': {col2.count()} records.")
except Exception as e:
    print(f"ERROR: Medical DB failed to load: {e}")

OPENAI_API_KEY is set (prefix: sk-proj-...)
User DB connection OK. Collection 'population_summaries': 82 records.
Medical DB connection OK. Collection 'medical_knowledge': 329 records.
